<a href="https://colab.research.google.com/github/juliaphaus/ds110/blob/main/Lecture31OtherWaysToSpeedUpCode.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Other ways to speed up code

"Cynthia, we did it!" Aubin said, walking into the room full of desktop machines.  "BND cases are falling, countries around the world are instituting our recommendations ... so why are you still in front of that computer after hours?"

"I'm still wondering why my code was so slow," she said, frowning before turning back to her monitor.  "Would you be willing to take a look?  I want this code to be faster next time -- assuming SAGE is still in the business of fighting epidemics."

"It certainly is," Aubin said with a grin.  "All right - let me tell you a few tricks they don't always teach you in school."

<!--*The code to find the individuals most in danger of contracting the deadly BND virus had still not finished.  "Go!  Go!  Go!" Cynthia said, banging her fist on the monitor, but that did not work, either.*

<i>"Did you try to get rid of nested loops to get the big-O down?" Aubin asked.</i>

<i>"Yes," Cynthia said sullenly.  "The Big-O can't get any better.  I have to look at all pairs of individuals, so it's just going to be $\Omega(N^2)$ no matter what."</i>

<i>"Is it running in parallel?"</i>

<i>"On a thousand different cores, yes."</i>

<i>"Did you try running a profiler?"</i>

*Cynthia's eyes widened.  "Of course, a profiler!"  She quickly started up a smaller test job.*

*Several minutes later, she smacked her forehead and buried her face in her hands.*

<i>"You left in the print statements?" Aubin said.</i>

<i>"I left in the print statements," Cynthia agreed.</i>-->

# Map

One of the tools for speeding up code that we'll talk about, parallelism, can make use of what are called map operations to do operations in parallel.  Before examining parallel maps, it's worthwhile to cover what map is normally.

A map operation takes a function as an argument and applies the function to everything in a collection, such as a list.  The function name is the first argument to map, and the collection (list) is the second argument.  The function is applied one at a time to every item in the list.

In [ ]:
def appendish(x):
  return x + "ish"

list(map(appendish, ["cool","interesting","neat"]))

map returns what's called a map object by default, which can be iterated over to get the values, but passing it to the list() function can turn it directly into a list.

In [ ]:
for i in map(appendish, ["cool","interesting","neat"]):
    print(i)

map is often used in conjunction with the lambda keyword, which can be used to define an anonymous function.  Rather than defining a function to apply ahead of time, the function can be defined in-place.  For example, lambda x: x + 2 defines an anonymous function that adds 2 to its argument.

In [ ]:
list(map(lambda x: x + "ish", ["cool","interesting","neat"]))

Put together, map and lambda have a similar effect to a list comprehension.  In Python, it's often more Pythonic to write a list comprehension than one of these lambda and map expressions.  But the pieces - map and lambda - get used elsewhere.

In [ ]:
[x + "ish" for x in ["cool","interesting","neat"]]

What makes map intriguing for speed purposes is, whatever it is doing to all the list elements, it could be doing in a very parallel way.  The elements don't interact at all, but are acted on totally separately; so there's no reason that can't happen simultaneously.  That opens the door to parallelism.

# Parallelism

Naturally, if your code is working on multiple processors, you can potentially get things done faster.  Most new machines these days now have multiple processors.  If you're not sure how many you're working with, you can ask with a function call from the module for using multiple processors, multiprocess.  (There is another module, multiprocessing, that is nearly identical but causes problems for notebooks.)



In [ ]:
# !pip install multiprocess

import multiprocess as mp
print(mp.cpu_count())

The key class used for multiprocessor work is Pool, an object that can be constructed with the number of processors as an argument.  Pool.map(function, iterable) will apply its function in parallel to all the items in the iterable -- or at least, in as parallel a way as possible given the number of processors.

In [ ]:
my_pool = mp.Pool(mp.cpu_count())

def is_even(x):
  return x % 2 == 0

result = my_pool.map(is_even, [5,6,7,8,9,10])
print(result)
my_pool.close()

The above method is synchronous, meaning the program can't proceed until all the results are in.  It's also possible to start asynchronous processes, where they work in the background until called for with get().

In [ ]:
my_pool = mp.Pool(mp.cpu_count())
result = my_pool.map_async(is_even, [5,6,7,8,9,10])
# We could do something else here
print('The jobs are working!')
print(result.get())

The jobs are working!
[False, True, False, True, False, True]


To create a more heavyweight parallel process, we can create an object that inherits from the multiprocessing.Process object, but override its run() method to do whatever we want.  The process can be started with .start() and we can wait for its results with join().

In [ ]:
import time
import os

class Process(mp.Process):
  def run(self):
    pid = os.getpid()
    time.sleep(1)
    print(f"Hello, multiprocessing from {pid}!")

p1 = Process()
p2 = Process()

p1.start()
p2.start()

p1.join()
p2.join()
print("We're done!")

Hello, multiprocessing from 1174!Hello, multiprocessing from 1175!

We're done!


Note through all this that there's some overhead in talking to the operating system to get the multiple processes set up, so there's no guarantee that small input sizes will see any speedup.

In [ ]:
def square(x):
  return x ** 2

# This will actually be slower because of the overhead of starting up processes
def distributed_squaring(n):
  my_pool = mp.Pool(mp.cpu_count())
  result = my_pool.map(square, range(n))
  my_pool.close()
  return result

%time distributed_squaring(10000)

def sequential_squaring(n):
  a = [x ** 2 for x in range(n)]
  return a

%time sequential_squaring(10000)

print("Done") # avoid printing the results

CPU times: user 15.8 ms, sys: 25.2 ms, total: 41 ms
Wall time: 41.2 ms
CPU times: user 297 μs, sys: 20 μs, total: 317 μs
Wall time: 318 μs
Done


# Vectorization



Vectorization is a way of optimizing away for loops in Python, which are slightly slower than they are in other languages.  If you can write the same code without saying "for," but instead treat what you're trying to do as a vector operation, the resulting code will probably be faster.  (Until the interpreter is optimized to do this automatically.)

As an example, we have on the one hand, the timing of a for loop that multiplies every element by 2 in an array, and on the other hand, the timing of a multiplication that realizes it's scaling the whole vector.

In [ ]:
import numpy as np

def for_loop_mult(n):
  original_list = np.array(range(n))
  for i in range(n):
    original_list[i] *= 2
%time for_loop_mult(12345678)

CPU times: user 2.57 s, sys: 395 ms, total: 2.96 s
Wall time: 1.66 s


In [ ]:
import numpy as np

def vectorized_mult(n):
  return np.array(range(n)) * 2

%time vectorized_mult(12345678)

CPU times: user 395 ms, sys: 42.9 ms, total: 438 ms
Wall time: 438 ms


array([       0,        2,        4, ..., 24691350, 24691352, 24691354])

Note again here that the gains aren't really guaranteed, and subtleties of the inner workings of Python could cause a non-vectorized version to run faster.

# Compilers

Python is an interpreted language, and interpreted languages are thought to be slow, as the interpreter typically needs to make sense of the programming language on-the-fly as the program is supposed to be running.  It's faster generally to compile the program ahead of time, interpreting it and turning it into machine code, assembly language, or something else low-level.

The most popular distribution of Python does compile the program ahead of time, though.  It's compiled into "bytecode," a similarity shared with Java, as a low-level set of instructions that can be somewhat optimized away from what the code literally says to do.  If the program has already been run once from the command line, then a .pyc file will be lingering as the compiled version of the program.

Since this compilation is always done, the main speed gain of having a .pyc file around comes from just loading the program without needing to compile it again.  The program will run just as fast after it's loaded if there was no pre-existing bytecode.



An advantage to using a compiler, as Python does automatically, is that the compiler can automatically detect when some code could be written more efficiently, and it can perform those optimizations.  This is why small optimizations for speed sometimes don't have the intended consequences.

The compilation into bytecode that I'm describing is done by CPython, the most popular distribution of Python, but there do exist other interpreters and compilers, and some could conceivably be faster than CPython's combined approach of compiliation and interpretation, especially for particular situations.

# Profiling

A profiler is a very important tool in making code run faster.  As we've discussed, the interpreter/compiler can make it very difficult to reason about where the slowest part of the code lies.  A profiler lets you determine with accuracy where the bottlenecks are, so you don't waste time trying to optimize the wrong thing.



Here, it tells us that getting a numpy array takes a significant amount of time in for_loop_mult().

In [ ]:
import cProfile
cProfile.run('for_loop_mult(12345678)')

         553 function calls (542 primitive calls) in 2.030 seconds

   Ordered by: standard name

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.593    0.593    0.593    0.593 2969500591.py:3(for_loop_mult)
        2    0.000    0.000    0.000    0.000 <frozen abc>:121(__subclasscheck__)
        5    0.000    0.000    0.000    0.000 <frozen importlib._bootstrap>:1390(_handle_fromlist)
        1    0.559    0.559    1.152    1.152 <string>:1(<module>)
        1    0.000    0.000    0.000    0.000 _base.py:337(_invoke_callbacks)
        1    0.000    0.000    0.000    0.000 _base.py:537(set_result)
        1    0.000    0.000    0.007    0.007 asyncio.py:200(_handle_events)
        1    0.000    0.000    0.000    0.000 asyncio.py:225(add_callback)
        5    0.000    0.000    0.000    0.000 attrsettr.py:42(__getattr__)
        5    0.000    0.000    0.000    0.000 attrsettr.py:65(_get_attr_opt)
        1    0.000    0.000    0.000    0.000 base_e

tottime doesn't sum over all functions called by a function, but cumtime does.  The "percall" next to each divides the adjacent time by the number of calls on the far left.  

For more on the profiler, see [the documentation.](https://docs.python.org/3/library/profile.html)

# Simple slow things

There are two kinds of operations which are generally rather slower than the others - printing things and asking for memory.  Both require some negotiation with the operating system, and so both will be slower than simple arithmetic operations.  (Other operating system business is also slow, such as asking for a socket in networking, but these are the two most common.)

In [ ]:
def count_and_print(n, pr):
  b = 0
  for i in range(n):
    if pr:
      print('I is now ' + str(i))  # Printing is slow, reads as socket send in cProfile
    b += 2
  return b

%time count_and_print(10000, True)

%time count_and_print(10000, False)



I is now 0
I is now 1
I is now 2
I is now 3
I is now 4
I is now 5
I is now 6
I is now 7
I is now 8
I is now 9
I is now 10
I is now 11
I is now 12
I is now 13
I is now 14
I is now 15
I is now 16
I is now 17
I is now 18
I is now 19
I is now 20
I is now 21
I is now 22
I is now 23
I is now 24
I is now 25
I is now 26
I is now 27
I is now 28
I is now 29
I is now 30
I is now 31
I is now 32
I is now 33
I is now 34
I is now 35
I is now 36
I is now 37
I is now 38
I is now 39
I is now 40
I is now 41
I is now 42
I is now 43
I is now 44
I is now 45
I is now 46
I is now 47
I is now 48
I is now 49
I is now 50
I is now 51
I is now 52
I is now 53
I is now 54
I is now 55
I is now 56
I is now 57
I is now 58
I is now 59
I is now 60
I is now 61
I is now 62
I is now 63
I is now 64
I is now 65
I is now 66
I is now 67
I is now 68
I is now 69
I is now 70
I is now 71
I is now 72
I is now 73
I is now 74
I is now 75
I is now 76
I is now 77
I is now 78
I is now 79
I is now 80
I is now 81
I is now 82
I is now 83
I 

20000

In [ ]:
def count_and_allocate(n, alloc):
  b = 0
  for i in range(n):
    if alloc:
      a = np.ones(n)
    b += 2
  return b

%time count_and_allocate(10000,True)

%time count_and_allocate(10000,False)

CPU times: user 16.8 ms, sys: 598 μs, total: 17.4 ms
Wall time: 17 ms
CPU times: user 180 μs, sys: 6 μs, total: 186 μs
Wall time: 188 μs


20000

# Final thoughts

Nothing shown here offers more than a constant-time speedup (except perhaps changes due to profiling), so it's worthwhile optimizing your approach from a big-O perspective first.  At that point, time the code using a profiler, optimize away very expensive operations, and use parallelism for anything that seems very parallel.